In [1]:
from pathlib import Path
import json
import pandas as pd

## Load the source data

The JSON file is our main source because its field names and values line up correctly. This cell turns the JSON records into a pandas table so we can inspect the data.


In [ ]:
json_path = Path("../Steam_Dataset/games.json")

# Use the JSON as the canonical source because its field names and values align.
with json_path.open("r", encoding="utf-8") as file:
    games = json.load(file)

sample = (
    pd.DataFrame.from_dict(games, orient="index")
    .rename_axis("AppID")
    .reset_index()
    .head(5_000)
)
sample.head()

,AppID,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,...,positive,negative,estimated_owners,average_playtime_forever,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu,tags
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0,0.00,0,,,,,...,0,0,0 - 0,0,0,0,0,0,0,[]
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0,5.24,0,"Springtime, April: when the cherry trees come ...","Springtime, April: when the cherry trees come ...","Spring has come, and our protagonist, Yukinari...",,...,252,3,0 - 20000,8,0,8,0,65,0,"{'Adventure': 27, 'Visual Novel': 19, 'Anime':..."
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0,4.99,0,"Immerse yourself in the most beloved, mystical...","Immerse yourself in the most beloved, mystical...",Discover an entrancing and spectacular world!,,...,21,3,0 - 20000,0,0,0,0,0,0,"{'Casual': 83, 'Card Game': 52, 'Solitaire': 4..."
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0,8.99,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...","synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",Yuha! I'll start the broadcast! Hakko's extrem...,,...,0,0,0 - 20000,0,0,0,0,0,1,[]
4,3631080,Maze Quest VR,"Apr 24, 2025",0,4.99,0,Its not just a Maze; its a Quest! Enter the ca...,Its not just a Maze; its a Quest! Enter the ca...,Its not just a Maze; its a Quest! Enter the ca...,,...,0,0,0 - 20000,0,0,0,0,0,0,[]


## First look at the table

Before changing anything, we check the size of the sample, the data types, and which fields are missing most often. This helps us understand what the dataset can realistically support.


In [3]:
sample.shape
sample.info()
sample.isna().mean().sort_values(ascending=False).head(15)

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 43 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   AppID                     5000 non-null   str    
 1   name                      5000 non-null   str    
 2   release_date              5000 non-null   str    
 3   required_age              5000 non-null   int64  
 4   price                     5000 non-null   float64
 5   dlc_count                 5000 non-null   int64  
 6   detailed_description      5000 non-null   str    
 7   about_the_game            5000 non-null   str    
 8   short_description         5000 non-null   str    
 9   reviews                   5000 non-null   str    
 10  header_image              5000 non-null   str    
 11  website                   5000 non-null   str    
 12  support_url               5000 non-null   str    
 13  support_email             5000 non-null   str    
 14  windows            

AppID                   0.0
name                    0.0
release_date            0.0
required_age            0.0
price                   0.0
dlc_count               0.0
detailed_description    0.0
about_the_game          0.0
short_description       0.0
reviews                 0.0
header_image            0.0
website                 0.0
support_url             0.0
support_email           0.0
windows                 0.0
dtype: float64

## Check zero values

A zero does not mean the same thing in every column. Here we check how often important numeric fields are zero so we can later decide which zeros are valid and which probably mean that data was not available.


In [4]:
numeric_columns = [
    "price",
    "required_age",
    "metacritic_score",
    "user_score",
    "positive",
    "negative",
    "recommendations",
    "average_playtime_forever",
    "peak_ccu",
]

zero_profile = (
    sample[numeric_columns]
    .eq(0)
    .mean()
    .sort_values(ascending=False)
    .rename("zero_rate")
)

zero_profile

user_score                  0.9996
required_age                0.9878
metacritic_score            0.9668
peak_ccu                    0.8458
recommendations             0.8278
average_playtime_forever    0.7970
negative                    0.4490
positive                    0.3432
price                       0.2054
Name: zero_rate, dtype: float64

## Make the numeric fields usable

Some values may be stored as text even though they represent numbers. We convert the fields used in calculations and turn values that cannot be converted into missing values instead of guessing.


In [5]:
numeric_columns = [
    "price",
    "metacritic_score",
    "positive",
    "negative",
    "recommendations",
    "average_playtime_forever",
    "peak_ccu",
]

for column in numeric_columns:
    sample[column] = pd.to_numeric(
        sample[column],
        errors="coerce"
    )

## Build a few useful review measures

The next cells create review count, positive review share, and simple availability flags. These are exploratory measures that make the raw fields easier to analyze.


In [6]:
sample["review_count"] = (
    sample["positive"] + sample["negative"]
)

sample["positive_ratio"] = (
    sample["positive"]
    / sample["review_count"].where(sample["review_count"] > 0)
)

sample["is_free"] = sample["price"].eq(0)

sample["has_metacritic"] = (
    sample["metacritic_score"] > 0
)

sample["has_playtime"] = (
    sample["average_playtime_forever"] > 0
)

sample[
    ["review_count", "positive_ratio"]
].describe()



,review_count,positive_ratio
count,5.000000e+03,3399.000000
mean,1.612340e+03,0.754646
std,3.986597e+04,0.243214
min,0.000000e+00,0.000000
25%,0.000000e+00,0.640000
50%,7.000000e+00,0.820896
75%,4.900000e+01,0.944444
max,2.498969e+06,1.000000


In [10]:
sample[
    ["is_free", "has_metacritic", "has_playtime"]
].mean().mul(100).round(1)

is_free           20.5
has_metacritic     3.3
has_playtime      20.3
dtype: float64

In [8]:
weighted_positive_ratio = (
    sample["positive"].sum() / sample["review_count"].sum()
)

weighted_positive_ratio

sample["review_count"].quantile(
    [0.50, 0.75, 0.90, 0.95, 0.99, 0.999]
)

sample[
    ["name", "review_count", "positive_ratio"]
].sort_values(
    "review_count",
    ascending=False
).head(10)

,name,review_count,positive_ratio
4193,Dota 2,2498969,0.815193
1111,Euro Truck Simulator 2,867404,0.973572
3692,PAYDAY 2,663082,0.896310
4117,Fall Guys,473743,0.813002
261,Battlefield™ 2042,297774,0.467422
1423,Monster Hunter Wilds,218415,0.582712
3231,American Truck Simulator,175287,0.968326
4107,Path of Exile 2,169021,0.701161
4238,Space Engineers,132347,0.885581
4029,Conan Exiles,107465,0.787000


## Check IDs and release dates

Here we make sure AppID can identify each record and check whether release dates can be converted into years. These checks support later trend analysis.


In [11]:
print("Rows:", len(sample))
print("Unique AppIDs:", sample["AppID"].nunique())
print("Duplicate AppIDs:", sample["AppID"].duplicated().sum())
print("Missing AppIDs:", sample["AppID"].isna().sum())

Rows: 5000
Unique AppIDs: 5000
Duplicate AppIDs: 0
Missing AppIDs: 0


In [12]:
release_text = (
    sample["release_date"]
    .astype("string")
    .str.strip()
)

sample["release_date_parsed"] = pd.to_datetime(
    release_text,
    errors="coerce",
    format="mixed"
)

sample["release_year"] = (
    sample["release_date_parsed"].dt.year
)

In [16]:
print(
    "Invalid or missing dates:",
    sample["release_date_parsed"].isna().sum()
)

sample[
    ["release_date", "release_date_parsed", "release_year"]
].head(10)



Invalid or missing dates: 0


,release_date,release_date_parsed,release_year
0,"Aug 1, 2023",2023-08-01,2023
1,"Jul 29, 2016",2016-07-29,2016
2,"May 6, 2019",2019-05-06,2019
3,"Oct 31, 2024",2024-10-31,2024
4,"Apr 24, 2025",2025-04-24,2025
5,"Apr 5, 2023",2023-04-05,2023
6,"Apr 8, 2025",2025-04-08,2025
7,"Feb 1, 2018",2018-02-01,2018
8,"May 7, 2021",2021-05-07,2021
9,"Dec 13, 2021",2021-12-13,2021


In [17]:
sample["release_date_parsed"].agg(["min", "max"])

min   2006-07-06
max   2025-12-19
Name: release_date_parsed, dtype: datetime64[us]

In [18]:
invalid_dates = sample[
    release_text.notna()
    & release_text.ne("")
    & sample["release_date_parsed"].isna()
]

invalid_dates[
    ["AppID", "name", "release_date"]
].head(20)

,AppID,name,release_date


## Understand the owner estimates

Owner counts are provided as ranges, not exact sales. We inspect the format, split each range into lower and upper bounds, and calculate a midpoint only as a rough comparison value.


In [19]:
owners_text = (
    sample["estimated_owners"]
    .astype("string")
    .str.strip()
)

print("Missing or empty values:", owners_text.isna().sum())
print("Unique owner ranges:", owners_text.nunique())

owners_text.value_counts().head(20)

Missing or empty values: 0
Unique owner ranges: 13


estimated_owners
0 - 20000                3089
0 - 0                     879
20000 - 50000             469
50000 - 100000            236
100000 - 200000           137
200000 - 500000            92
500000 - 1000000           54
1000000 - 2000000          25
2000000 - 5000000           9
20000000 - 50000000         4
5000000 - 10000000          3
10000000 - 20000000         2
100000000 - 200000000       1
Name: count, dtype: Int64

In [20]:
owner_parts = owners_text.str.extract(
    r"^\s*([\d,]+)\s*-\s*([\d,]+)\s*$"
)

owner_parts.columns = [
    "owners_lower_text",
    "owners_upper_text",
]

owner_parts.head(10)

,owners_lower_text,owners_upper_text
0,0,0
1,0,20000
2,0,20000
3,0,20000
4,0,20000
5,0,20000
6,0,20000
7,0,20000
8,0,20000
9,0,20000


In [21]:
invalid_owner_ranges = sample[
    owner_parts.isna().any(axis=1)
]

print("Invalid owner ranges:", len(invalid_owner_ranges))

invalid_owner_ranges[
    ["AppID", "name", "estimated_owners"]
].head(20)

Invalid owner ranges: 0


,AppID,name,estimated_owners


In [22]:
sample["owners_lower"] = pd.to_numeric(
    owner_parts["owners_lower_text"]
    .str.replace(",", "", regex=False),
    errors="coerce"
).astype("Int64")

sample["owners_upper"] = pd.to_numeric(
    owner_parts["owners_upper_text"]
    .str.replace(",", "", regex=False),
    errors="coerce"
).astype("Int64")

sample["owners_midpoint"] = (
    sample["owners_lower"] + sample["owners_upper"]
) / 2

In [23]:
sample[
    [
        "estimated_owners",
        "owners_lower",
        "owners_upper",
        "owners_midpoint",
    ]
].head(10)

,estimated_owners,owners_lower,owners_upper,owners_midpoint
0,0 - 0,0,0,0.0
1,0 - 20000,0,20000,10000.0
2,0 - 20000,0,20000,10000.0
3,0 - 20000,0,20000,10000.0
4,0 - 20000,0,20000,10000.0
5,0 - 20000,0,20000,10000.0
6,0 - 20000,0,20000,10000.0
7,0 - 20000,0,20000,10000.0
8,0 - 20000,0,20000,10000.0
9,0 - 20000,0,20000,10000.0


In [24]:
invalid_owner_logic = sample[
    (sample["owners_lower"] > sample["owners_upper"])
    | sample["owners_lower"].isna()
    | sample["owners_upper"].isna()
]

print("Invalid owner records:", len(invalid_owner_logic))

Invalid owner records: 0


In [25]:
sample[
    [
        "owners_lower",
        "owners_upper",
        "owners_midpoint",
    ]
].describe()

,owners_lower,owners_upper,owners_midpoint
count,5000.0,5000.0,5000.0
mean,67656.0,160246.0,113951.0
std,1544980.769712,3206060.917201,2373726.978941
min,0.0,0.0,0.0
25%,0.0,20000.0,10000.0
50%,0.0,20000.0,10000.0
75%,0.0,20000.0,10000.0
max,100000000.0,200000000.0,150000000.0


## Inspect multi-value fields

Developers, publishers, genres, categories, and tags can contain several values for one game. We inspect their structure before deciding how they should be represented in the cleaned data.


In [26]:
list_columns = [
    "developers",
    "publishers",
    "genres",
    "categories",
    "tags",
]

for column in list_columns:
    print(f"\n--- {column} ---")
    print(sample[column].head(3).tolist())


--- developers ---
[[], ['minori'], ['Somer Games']]

--- publishers ---
[[], ['MangaGamer'], ['8floor']]

--- genres ---
[[], ['Adventure'], ['Casual']]

--- categories ---
[[], ['Single-player', 'Steam Trading Cards', 'Steam Cloud', 'Family Sharing'], ['Single-player', 'Family Sharing']]

--- tags ---
[[], {'Adventure': 27, 'Visual Novel': 19, 'Anime': 19, 'Cute': 7}, {'Casual': 83, 'Card Game': 52, 'Solitaire': 47, 'Puzzle': 43, 'Hidden Object': 40, '2D': 34, 'Colorful': 32, 'Stylized': 30, 'Logic': 28, 'Mystery': 26, 'Atmospheric': 24, 'Family Friendly': 22, 'PvE': 20, 'Tutorial': 18, 'Singleplayer': 16, 'Tabletop': 14}]


In [27]:
def list_length(value):
    if isinstance(value, list):
        return len(value)
    return 0


nested_profile = {}

for column in list_columns:
    lengths = sample[column].apply(list_length)

    nested_profile[column] = {
        "non_empty_count": (lengths > 0).sum(),
        "non_empty_pct": (lengths > 0).mean() * 100,
        "average_items_when_non_empty": (
            lengths.where(lengths > 0).mean()
        ),
        "maximum_items": lengths.max(),
        "unique_values": sample[column].explode().dropna().nunique(),
    }

pd.DataFrame(nested_profile).T.round(2)

,non_empty_count,non_empty_pct,average_items_when_non_empty,maximum_items,unique_values
developers,4672.0,93.44,1.10,12.0,4733.0
publishers,4654.0,93.08,1.05,3.0,4121.0
genres,4670.0,93.40,2.86,16.0,27.0
categories,4658.0,93.16,4.48,25.0,55.0
tags,0.0,0.00,NaN,0.0,441.0


In [28]:
tag_lengths = sample["tags"].apply(
    lambda value: len(value)
    if isinstance(value, dict)
    else 0
)

print("Games with tags:", (tag_lengths > 0).sum())
print("Percentage with tags:", round((tag_lengths > 0).mean() * 100, 2))
print("Average tags when non-empty:", tag_lengths.where(tag_lengths > 0).mean())
print("Maximum tags on one game:", tag_lengths.max())

Games with tags: 3420
Percentage with tags: 68.4
Average tags when non-empty: 14.16140350877193
Maximum tags on one game: 21


In [29]:
tag_votes = {}

for tag_dictionary in sample["tags"]:
    if isinstance(tag_dictionary, dict):
        for tag, votes in tag_dictionary.items():
            tag_votes[tag] = tag_votes.get(tag, 0) + votes

pd.Series(tag_votes).sort_values(
    ascending=False
).head(20)

Action            202426
Adventure         190355
Singleplayer      184535
Casual            168850
2D                137822
Indie             135331
RPG               118555
3D                115310
Simulation        106978
Strategy          104048
Exploration        94618
Atmospheric        93284
Colorful           92954
Puzzle             89324
Multiplayer        87300
Pixel Graphics     78084
Cute               77974
Relaxing           76368
Story Rich         76174
Free to Play       73717
dtype: int64

In [30]:
list_columns = [
    "developers",
    "publishers",
    "genres",
    "categories",
]

for column in list_columns:
    print(f"\n--- Top {column} ---")

    values = (
        sample[column]
        .explode()
        .dropna()
        .value_counts()
        .head(15)
    )

    print(values)


--- Top developers ---
developers
EroticGamesClub             13
Boogygames Studios           9
Laush Dmitriy Sergeevich     8
Cyber Keks                   8
Cute Hannah's Games          8
Tero Lunkka                  8
Feral Interactive (Mac)      7
Gamesforgames                7
RewindApp                    6
SCS Software                 6
NanningsGames                6
Creobit                      6
Valkeala Software            6
Somer Games                  5
Eidos-Montréal               5
Name: count, dtype: int64

--- Top publishers ---
publishers
8floor                             17
BFG Entertainment                  16
EroticGamesClub                    13
Kagura Games                       12
Laush Studio                       12
PlayWay S.A.                       10
My Way Games                       10
Atari                               9
DigiPen Institute of Technology     9
Boogygames Studios                  9
Strategy First                      9
Cyber Keks           

## Look at distributions and finish the data checks

These cells show the spread of the main metrics, release-year coverage, platform support, and the remaining field-quality issues. The goal is to understand the data before making cleaning decisions.


In [31]:
metric_columns = [
    "price",
    "owners_midpoint",
    "peak_ccu",
    "review_count",
    "average_playtime_forever",
    "positive_ratio",
]

sample[metric_columns].quantile(
    [0, 0.25, 0.50, 0.75, 0.90, 0.99, 1.00]
).T

,0.00,0.25,0.50,0.75,0.90,0.99,1.00
price,0.0,0.59,1.99,4.99,9.99,29.99,199.99
owners_midpoint,0.0,10000.0,10000.0,10000.0,75000.0,750000.0,150000000.0
peak_ccu,0.0,0.0,0.0,0.0,2.0,176.02,623941.0
review_count,0.0,0.0,7.0,49.0,336.0,13221.59,2498969.0
average_playtime_forever,0.0,0.0,0.0,0.0,218.2,2136.17,54064.0
positive_ratio,0.0,0.64,0.820896,0.944444,1.0,1.0,1.0


In [ ]:
sample["release_year"].value_counts().sort_index()

In [32]:
platform_columns = [
    "windows",
    "mac",
    "linux",
]

platform_summary = pd.DataFrame({
    "available_count": sample[platform_columns].sum(),
    "available_pct": sample[platform_columns].mean().mul(100),
})

platform_summary.round(1)

,available_count,available_pct
windows,4998,100.0
mac,845,16.9
linux,616,12.3


In [33]:
year_counts = (
    sample["release_year"]
    .value_counts()
    .sort_index()
)

print("Earliest year:", year_counts.index.min())
print("Latest year:", year_counts.index.max())

print("\nEarliest years:")
print(year_counts.head(10))

print("\nLatest years:")
print(year_counts.tail(10))

Earliest year: 2006
Latest year: 2025

Earliest years:
release_year
2006      1
2007      5
2008      7
2009     12
2010     12
2011     11
2012     12
2013     26
2014     62
2015    101
Name: count, dtype: int64

Latest years:
release_year
2016     175
2017     216
2018     310
2019     278
2020     377
2021     458
2022     495
2023     604
2024     811
2025    1027
Name: count, dtype: int64


In [34]:
future_years = sample[
    sample["release_year"] > pd.Timestamp.now().year
]

print("Future-dated records:", len(future_years))

future_years[
    ["AppID", "name", "release_date", "release_year"]
].head(20)

Future-dated records: 0


,AppID,name,release_date,release_year


In [35]:
scalar_columns = [
    "AppID",
    "name",
    "release_date",
    "release_date_parsed",
    "release_year",
    "estimated_owners",
    "owners_lower",
    "owners_upper",
    "owners_midpoint",
    "price",
    "required_age",
    "metacritic_score",
    "user_score",
    "positive",
    "negative",
    "review_count",
    "positive_ratio",
    "recommendations",
    "average_playtime_forever",
    "average_playtime_2weeks",
    "median_playtime_forever",
    "median_playtime_2weeks",
    "peak_ccu",
    "windows",
    "mac",
    "linux",
]

field_profile = pd.DataFrame({
    "dtype": [
        str(sample[column].dtype)
        for column in scalar_columns
    ],
    "missing_pct": [
        sample[column].isna().mean() * 100
        for column in scalar_columns
    ],
    "unique_values": [
        sample[column].nunique(dropna=True)
        for column in scalar_columns
    ],
}, index=scalar_columns)

field_profile.sort_values(
    "missing_pct",
    ascending=False
).round(1)

,dtype,missing_pct,unique_values
positive_ratio,float64,32.0,1301
AppID,str,0.0,5000
release_date,str,0.0,2474
release_date_parsed,datetime64[us],0.0,2474
release_year,int32,0.0,20
estimated_owners,str,0.0,13
owners_lower,Int64,0.0,12
owners_upper,Int64,0.0,13
owners_midpoint,Float64,0.0,13
name,str,0.0,4994


In [36]:
duplicate_names = sample[
    sample["name"].duplicated(keep=False)
].sort_values("name")

duplicate_names[
    ["AppID", "name", "release_date"]
]

,AppID,name,release_date
1202,3437200,LUNA,"Sep 24, 2025"
3934,1450100,LUNA,"Nov 20, 2020"
499,2386620,Labyrinth,"Sep 14, 2023"
4282,412310,Labyrinth,"Mar 7, 2016"
997,1122690,Last Stop,"Jul 22, 2021"
2411,2327980,Last Stop,"Mar 13, 2023"
67,849178,Shadow of the Tomb Raider: Definitive Edition,"Sep 14, 2018"
1179,849165,Shadow of the Tomb Raider: Definitive Edition,"Sep 14, 2018"
1608,750920,Shadow of the Tomb Raider: Definitive Edition,"Sep 14, 2018"
3621,849163,Shadow of the Tomb Raider: Definitive Edition,"Sep 14, 2018"


In [38]:
for column in [
    "user_score",
    "metacritic_score",
    "required_age",
]:
    print(f"\n--- {column} ---")
    print(sample[column].value_counts().sort_index())


--- user_score ---
user_score
0      4998
76        1
100       1
Name: count, dtype: int64

--- metacritic_score ---
metacritic_score
0     4834
46       1
50       1
53       1
54       2
57       2
58       2
59       3
60       3
61       2
62       4
63       3
64       3
65       5
66       7
67       3
68       4
69       3
70       8
71       4
72       7
73       5
74       4
75       8
76       6
77      12
78       9
79      11
80      11
81       7
82       2
83       5
84       3
85       2
86       2
87       1
89       3
90       5
91       2
Name: count, dtype: int64

--- required_age ---
required_age
0     4939
12       2
13       7
17      47
18       5
Name: count, dtype: int64


## Reconnaissance summary

This first pass gave me a clearer picture of the dataset before cleaning it. I am using the JSON file because the CSV has shifted column labels. AppID is a reliable key, but game names are not unique.

Release dates parsed successfully. Estimated owners are ranges, so the lower bound, upper bound, and midpoint should be treated as estimates rather than exact sales. Reviews, owners, peak CCU, and playtime are heavily skewed, and many zero values probably mean that information was unavailable rather than that the true value was zero.

Developers, publishers, genres, categories, and tags contain multiple values. Tags also include vote counts, so these fields will need special handling in the cleaned dataset.

The first 5,000 records were useful for learning the structure and identifying data-quality issues, but they should not be used for final conclusions about the full Steam catalog.

The next stage will use the full JSON data to standardize fields, document decisions about missing and zero values, and normalize the multi-value fields.